# Can we predict a warranty claim within a year?
Target: `claim_within_year`. Follows `checklist.md`.

Data notes: no date column and every `device_serial` is unique (one row per device), so there is no time order or repeated entity to respect; a stratified random split is the appropriate hold-out. `device_serial` is an identifier and is dropped. All remaining columns (channel, device age, price, reported daily use) are recorded at sale/registration, i.e. known at prediction time.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
SEED = 0
df = pd.read_csv('data.csv')
print(df.shape, df.device_serial.is_unique, df.isna().sum().sum())
df.claim_within_year.mean()

## 1. Test set, set aside before any modelling

In [ ]:
y = df['claim_within_year']
X = df.drop(columns=['claim_within_year', 'device_serial'])   # identifier dropped
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
print(len(X_train), len(X_test), y_train.mean(), y_test.mean())

## 2. Candidates (named before scoring)
1. **Baseline**: predicts the training prevalence for every row.
2. **Logistic regression** (scaled numerics, one-hot channel), grid over `C`.
3. **Gradient boosting** (`HistGradientBoostingClassifier`), grid over learning rate / depth / leaf size.

Selection metric: ROC AUC under 5-fold stratified CV on the training rows only (positives are ~13%, so PR AUC is also reported). Preprocessing lives inside the pipelines so it is fit per fold. The threshold is chosen on out-of-fold predictions to maximise F1.

In [ ]:
num = ['device_age_years', 'price_eur', 'daily_hours']; cat = ['channel']
pre = lambda: ColumnTransformer([('n', StandardScaler(), num), ('c', OneHotEncoder(handle_unknown='ignore'), cat)])
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
candidates = {
    'baseline': (Pipeline([('p', pre()), ('m', DummyClassifier(strategy='prior'))]), {}),
    'logreg': (Pipeline([('p', pre()), ('m', LogisticRegression(max_iter=1000))]), {'m__C': [0.01, 0.1, 1, 10]}),
    'hgb': (Pipeline([('p', pre()), ('m', HistGradientBoostingClassifier(random_state=SEED))]),
            {'m__learning_rate': [0.03, 0.1], 'm__max_depth': [2, 3, None], 'm__min_samples_leaf': [20, 100]}),
}
searches = {}
for name, (pipe, grid) in candidates.items():
    gs = GridSearchCV(pipe, grid or {}, scoring='roc_auc', cv=cv, refit=True).fit(X_train, y_train)
    searches[name] = gs
    print(f'{name:9s} CV AUC {gs.best_score_:.4f}  {gs.best_params_}')

In [ ]:
# Out-of-fold predictions on training rows only (hyperparameters were picked on these folds, so slightly optimistic)
oof = {n: cross_val_predict(g.best_estimator_, X_train, y_train, cv=cv, method='predict_proba')[:, 1] for n, g in searches.items()}
for n, p in oof.items():
    print(n, 'OOF AUC %.4f  PR-AUC %.4f' % (roc_auc_score(y_train, p), average_precision_score(y_train, p)))
best = max(searches, key=lambda n: searches[n].best_score_)
ths = np.linspace(0.02, 0.6, 59)
f1s = [f1_score(y_train, oof[best] >= t) for t in ths]
thr = ths[int(np.argmax(f1s))]
print('chosen model:', best, searches[best].best_params_, 'threshold %.2f' % thr, 'OOF F1 %.3f' % max(f1s))

## 3. Choice (recorded before the test set is touched)
Based on CV alone: **logistic regression, C=0.1** (CV AUC 0.716 vs 0.712 for gradient boosting, 0.500 for the baseline), decision threshold **0.18** (maximises out-of-fold F1 = 0.36). Features: channel, device_age_years, price_eur, daily_hours. The test set is scored once, in the next cell.


In [ ]:
final = searches[best].best_estimator_.fit(X_train, y_train)   # refit on all training rows
p = final.predict_proba(X_test)[:, 1]
base = searches['baseline'].best_estimator_.fit(X_train, y_train).predict_proba(X_test)[:, 1]
yh = p >= thr
print('TEST (scored once)')
print('ROC AUC %.4f (baseline %.4f)' % (roc_auc_score(y_test, p), roc_auc_score(y_test, base)))
print('PR AUC  %.4f (prevalence %.4f)' % (average_precision_score(y_test, p), y_test.mean()))
print('@thr %.2f  precision %.3f recall %.3f F1 %.3f' % (thr, precision_score(y_test, yh), recall_score(y_test, yh), f1_score(y_test, yh)))

## 4. Conclusion
Claims are predictable to a moderate degree from sale-time information. On the held-out 20% (scored once), the chosen logistic regression reaches ROC AUC 0.761 (baseline 0.500) and PR AUC 0.343 against a 12.6% prevalence. At the CV-chosen threshold of 0.18 it catches about half of the claims (recall 0.51) with precision 0.32, roughly 2.5x the base rate. That is useful for ranking or prioritising, not for confident individual calls.

Caveats: the test AUC is higher than the CV AUC (0.716); with 1,400 test rows (~176 positives) the test estimate is noisy, so the CV figure is the more conservative expectation. There is no date column, so the split is random and cannot show how the model holds up on future sales. Gradient boosting did not beat logistic regression, which suggests the signal is mostly simple and monotone.
